# Model Training Session

This notebook demonstrates how to configure and run an end-to-end continuous training session.

**Note**:

At this point if all cells in `01_data_preparation.ipynb` have already been run, we should have artifacts ready for training under `path/to/this/notebook/experiment`.

Otherwise either point `prepared_dpath` to where your actual data was prepared, or run the `01_data_preparation.ipynb` end-to-end to generate synthetic data for the following cell to demo a training session

## Environment Setup & Imports

> **Prerequisite**: Ensure this notebook is attached to a cluster with the `landseg` wheel pre-installed (e.g. Databricks compute via cluster init script or cluster libraries) or an active local virtual environment. No in-notebook pip installation is required.

In [ ]:
# standard imports
import pathlib
import sys
# local imports
# Attempt importing landseg directly from site-packages (wheel install).
# Fall back to local '../src' only when running inside the repository clone.
try:
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.notebooks as nb
except ImportError:
    repo_src = pathlib.Path.cwd().parent / 'src'
    if repo_src.exists():
        sys.path.insert(0, str(repo_src))
    import landseg.adapters.api as api
    import landseg.mlops as mlops
    import landseg.notebooks as nb

## Parameter Widgets Setup

Define interactive configuration widgets for training parameterization, output storage location, and Unity Catalog model registration.

In [ ]:
# Ergonomic configuration widgets for interactive training parameterization
widgets = nb.NotebookWidgets()

# I/O destinations
# Leave default empty so users designate a specific UC Volume or directory before training
output_dpath = widgets.text('output_root', default='', label='Output Root Directory')
ingested_dpath = widgets.text('ingest_dpath', default='', label='Ingested Data Blocks Directory')
prepared_dpath = widgets.text('prepare_dpath', default='', label='Prepared Data Directory')
exp_root = widgets.text('exp_root', default='../experiment', label='Experiment Root Directory')

# Training parameters
batch_size = widgets.int('batch_size', default=16, label='Batch Size')
patch_size = widgets.int('patch_size', default=128, label='Patch Size (px)')
max_epochs = widgets.int('max_epochs', default=5, label='Max Epochs')
model_body = widgets.dropdown('model_body', default='unet', choices=['unet', 'unetpp', 'unetppp'], label='Model Body')
device = widgets.dropdown('device', default='auto', choices=['auto', 'cuda', 'cpu'], label='Execution Device')

# Unity Catalog model registration
register_model = widgets.boolean('register_model', default=False, label='Register to Unity Catalog')
catalog = widgets.text('catalog', default='main', label='UC Catalog')
schema = widgets.text('schema', default='landcover', label='UC Schema')
model_name = widgets.text('model_name', default='landseg_unet', label='Model Name')

## Validate Inputs & Output Destination

Verify required parameters and test destination directory writability before committing compute.
On Databricks, this cell halts execution if `output_root` is unconfigured, preventing unreviewed training runs.

In [ ]:
if mlops.is_databricks():
    resolved_ingested_dpath = nb.require_widget(
        'ingest_dpath',
        error_message=(
            'Set the "ingest_dpath" widget to a Unity Catalog Volume path '
            '(e.g. "/Volumes/<catalog>/<schema>/<volume>/<dir>") or local directory, '
            'then re-run the notebook.'
        ),
        widgets=widgets,
    )

    resolved_prepared_dpath = nb.require_widget(
        'prepare_dpath',
        error_message=(
            'Set the "prepare_dpath" widget to a Unity Catalog Volume path '
            '(e.g. "/Volumes/<catalog>/<schema>/<volume>/<dir>") or local directory, '
            'then re-run the notebook.'
        ),
        widgets=widgets,
    )

    resolved_output_dpath = nb.require_widget(
        'output_root',
        validator=mlops.validate_output_root,
        error_message=(
            'Set the "output_root" widget to a Unity Catalog Volume path '
            '(e.g. "/Volumes/<catalog>/<schema>/<volume>/<dir>") or local directory, '
            'then re-run the notebook.'
        ),
        widgets=widgets,
    )

    print('Running on Databricks with resolved I/O')
    print(f'Upstream ingested data at {resolved_ingested_dpath}')
    print(f'Upstream prepared data at {resolved_prepared_dpath}')
    print(f'Training results output at {resolved_output_dpath}')

else:
    resolved_ingested_dpath = pathlib.Path(exp_root) / 'artifacts' / 'ingested_data'
    resolved_prepared_dpath = pathlib.Path(exp_root) / 'artifacts' / 'prepared_data'
    resolved_output_dpath = pathlib.Path(exp_root) / 'results'
    print(f'Running local with expected I/O root at {exp_root}')
    print(f'Upstream ingested data at {resolved_ingested_dpath}')
    print(f'Upstream prepared data at {resolved_prepared_dpath}')
    print(f'Training results output at {resolved_output_dpath}')

resolved_output_dpath.mkdir(exist_ok=True)

## Basic Training Configurations (a continuous training session)

In [ ]:
# init configurator with canonical destination and upstream paths
configurator = api.TrainingSessionConfigurator(
    ingestion_dpath=resolved_ingested_dpath,
    preparation_dpath=resolved_prepared_dpath,
    output_dpath=resolved_output_dpath,
)

# set data loading parameters
configurator.set_data_loading(
    batch_size=batch_size,
    patch_size=patch_size,
)

# set domain source for conditioning
configurator.set_domain_source(
    category_domain=None,
    continuous_domain=None,
)

# set model architecture
configurator.set_model(
    body=model_body,
    bottleneck='conv',
    base_channel=16,
)

# set optimization
configurator.set_optimization(
    optimizer='AdamW',
    learning_rate=1e-4,
    weight_decay=1e-3,
    scheduler='CosAnneal',
)

# set engine tasks related knobs
configurator.set_tasks(
    logit_adjust_alpha=0.0,
    exclude_classes=None,
    loss_weights={
        'focal': 0.5,
        'dice': 0.5,
        'spectral': 1e-3,
        'total_var': 1e-4,
        'ecological': 0.0,
    },
)

# set runtime parameters
configurator.set_runtime(
    max_epochs=max_epochs,
    active_heads=None,
    patience_epoch=None,
    track_heads=None,
)

# configure target device if specified
configurator.set_device(device)

# run pre-flight readiness check (audits dataset manifest, GPU/CUDA VRAM headroom, model backbone)
configurator.preflight()

## Run Training Session

In [ ]:
# run the training session with graceful cancellation and GPU memory protection
with nb.safe_training_session(resolved_output_dpath):
    api.run(configurator.running_root_config)

## In-Cell Training Telemetry & Metric Curves

Plot and inspect real-time training loss vs. validation loss trajectories and tracking mIoU curves.

In [ ]:
# Plot and inspect training loss and validation metric curves
try:
    fig = nb.plot_training_curves(resolved_output_dpath)
except (FileNotFoundError, ValueError) as err:
    print(f'Training metrics preview unavailable: {err}')

## Qualitative Inspection of Segmentation Predictions

Render side-by-side visual comparisons of input RGB imagery, ground truth masks, and model predictions.

In [ ]:
# Render qualitative prediction preview (RGB Imagery, Ground Truth Mask, Model Prediction)
fig = nb.render_prediction_preview(resolved_output_dpath)

## Unity Catalog Model Registration (Databricks)

Publish validated model checkpoints and governance metadata directly to the Unity Catalog Model Registry (models:/<catalog>.<schema>.<model_name>).
When executed outside Databricks or in headless CI, this cell automatically detects the environment and safely logs an inspection summary without failing.

In [ ]:
# Publish validated model to Unity Catalog Model Registry
if register_model:
    registered_version = mlops.register_unity_catalog_model(
        exp_root=resolved_output_dpath,
        catalog=catalog,
        schema=schema,
        model_name=model_name,
    )
else:
    print('Unity Catalog registration bypassed (set register_model=True to publish).')